In [0]:
CATALOG = dbutils.widgets.get("catalog")
GOLD = f"{CATALOG}.skybound_gold"
SILVER_AIRPORTS = f"{CATALOG}.skybound_silver.airports_enriched_silver"

In [0]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {GOLD}.dim_airport AS
  SELECT
    ident AS airport_icao,
    airport_name,
    country_name,
    continent,
    country_code,
    iso_region,
    municipality,
    latitude_deg,
    longitude_deg,
    elevation_ft,
    airport_type,
    CURRENT_TIMESTAMP() AS updated_at
  FROM {SILVER_AIRPORTS}
  WHERE country_name != 'Russia'
    AND airport_type != 'heliport'
""")

In [0]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {GOLD}.dim_date AS
  SELECT
    date_format(ts, 'yyyyMMddHH')        AS date_hour_sk,   -- surrogate key
    CAST(ts AS DATE)                      AS obs_date,
    date_format(ts, 'yyyy-MM-dd HH:00')  AS obs_hour_label,
    hour(ts)                             AS hour_of_day,
    dayofweek(ts)                        AS day_of_week,
    month(ts)                            AS month_num,
    year(ts)                             AS year_num,
    CASE WHEN hour(ts) BETWEEN 6 AND 21 THEN 'day' ELSE 'night' END AS day_night
  FROM (
    SELECT explode(sequence(
      TIMESTAMP '2026-03-01 00:00:00',
      TIMESTAMP '2026-05-01 23:00:00',
      INTERVAL 1 HOUR
    )) AS ts
  )
""")

In [0]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {GOLD}.dim_flight_category AS
  SELECT * FROM VALUES
    (1, 'VFR', 'Visual Flight Rules', 'green', False),
    (2, 'MVFR', 'Marginal Visual Flight Rules', 'blue', False),
    (3, 'IFR', 'Instrument Flight Rules', 'red', True),
    (4, 'LIFR', 'Low Instrument Flight Rules', 'magenta',True)
  AS t(category_sk, code, name, map_color, is_ifr)
""")